# **Hospital Treatment Cost Predictor**

Clean Streamlit GUI for the saved Random Forest model.

Run the cells from top to bottom in Google Colab.

In [1]:
# Install the same scikit-learn version used when the model was saved.
!pip install -q streamlit joblib pandas numpy scikit-learn==1.6.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.5 MB/s eta 0:00:00


In [2]:
import os

if not os.path.exists('model_file.joblib'):
    from google.colab import files
    print('Upload model_file.joblib')
    files.upload()
else:
    print('model_file.joblib is already available.')


Upload model_file.joblib


Saving model_file.joblib to model_file.joblib


In [3]:
import joblib

bundle = joblib.load('model_file.joblib')

print('Model:', bundle['model_name'])
print('Number of features:', len(bundle['feature_columns']))
print('Test size:', bundle['test_size'])
print('Random state:', bundle['random_state'])


Model: Random Forest Regressor
Number of features: 30
Test size: 0.2
Random state: 42


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib

MODEL_FILE = "model_file.joblib"

bundle = joblib.load(MODEL_FILE)
model = bundle["model"]

st.set_page_config(
    page_title="Hospital Treatment Cost Predictor",
    page_icon="🏥",
    layout="centered",
)

st.title("🏥 Hospital Treatment Cost Predictor")
st.write("Enter patient information to predict the treatment cost dynamically using our Random Forest model.")
st.divider()

# Organise the inputs into two neat columns to look professional
col1, col2 = st.columns(2)

with col1:
    st.markdown("### 👤 Patient Profile")
    age = st.number_input(
        "Age",
        min_value=int(bundle["domain_bounds"]["Age"][0]),
        max_value=int(bundle["domain_bounds"]["Age"][1]),
        value=30,
        step=1,
        help="The age of the patient in years."
    )

    gender = st.selectbox(
        "Gender",
        bundle["gender_encoder"].classes_.tolist(),
        help="Biological sex at birth."
    )

    insurance = st.selectbox(
        "Insurance Type",
        bundle["onehot_encoder"].categories_[2].tolist(),
        help="Primary provider of the health insurance coverage."
    )

    readmission = st.selectbox(
        "Readmission Flag",
        options=[0, 1],
        format_func=lambda x: "Yes" if x == 1 else "No",
        help="Has the patient been readmitted within 30 days?"
    )

with col2:
    st.markdown("### 🩺 Admission Details")
    department = st.selectbox(
        "Department",
        bundle["onehot_encoder"].categories_[0].tolist(),
        help="The admitting medical department."
    )

    diagnosis = st.selectbox(
        "Diagnosis",
        bundle["onehot_encoder"].categories_[1].tolist(),
        help="Primary medical diagnosis."
    )

    severity = st.selectbox(
        "Severity Level",
        bundle["severity_encoder"].categories_[0].tolist(),
        help="Overall triage or emergency level assigned."
    )

    length_of_stay = st.number_input(
        "Length of Stay (Days)",
        min_value=int(bundle["domain_bounds"]["Length_of_Stay_Days"][0]),
        max_value=int(bundle["domain_bounds"]["Length_of_Stay_Days"][1]),
        value=3,
        step=1,
        help="Total duration of the hospital stay in days."
    )

    wait_time = st.number_input(
        "Wait Time (Minutes)",
        min_value=int(bundle["domain_bounds"]["Wait_Time_Minutes"][0]),
        max_value=int(bundle["domain_bounds"]["Wait_Time_Minutes"][1]),
        value=30,
        step=1,
        help="Pre-admission triage wait time."
    )

outcome = st.selectbox(
    "Discharge Outcome",
    bundle["onehot_encoder"].categories_[3].tolist(),
    help="Medical status of the patient at discharge."
)

st.divider()

if st.button("Predict Treatment Cost", type="primary", use_container_width=True):
    try:
        input_data = pd.DataFrame([{
            "Age": age,
            "Gender": gender,
            "Department": department,
            "Diagnosis": diagnosis,
            "Severity_Level": severity,
            "Length_of_Stay_Days": length_of_stay,
            "Wait_Time_Minutes": wait_time,
            "Insurance_Type": insurance,
            "Readmission_Flag": readmission,
            "Outcome": outcome,
        }])

        # Apply the exact same saved preprocessing used during training.
        X_input = input_data.copy()

        numeric_cols = bundle["numeric_cols"]
        categorical_cols = bundle["categorical_cols"]
        nominal_cols = bundle["nominal_cols"]

        X_input[numeric_cols] = bundle["numeric_imputer"].transform(
            X_input[numeric_cols]
        )
        X_input[categorical_cols] = bundle["categorical_imputer"].transform(
            X_input[categorical_cols]
        )

        X_input[["Severity_Level"]] = bundle["severity_encoder"].transform(
            X_input[["Severity_Level"]]
        )

        X_input["Gender"] = bundle["gender_encoder"].transform(
            X_input["Gender"]
        )

        onehot = pd.DataFrame(
            bundle["onehot_encoder"].transform(X_input[nominal_cols]),
            columns=bundle["onehot_encoder"].get_feature_names_out(nominal_cols),
            index=X_input.index,
        )

        X_input = pd.concat(
            [X_input.drop(columns=nominal_cols), onehot],
            axis=1,
        )

        X_input[numeric_cols] = bundle["scaler"].transform(
            X_input[numeric_cols]
        )

        # Use the exact feature order from training.
        X_input = X_input[bundle["feature_columns"]]

        prediction = model.predict(X_input)

        if bundle["uses_target_scaling"]:
            prediction = bundle["target_scaler"].inverse_transform(
                prediction.reshape(-1, 1)
            ).ravel()

        prediction = float(prediction[0])

        st.markdown("### 💰 Estimated Results")
        st.metric(label="Predicted Treatment Cost", value=f"${prediction:,.2f}")
        st.success("Prediction generated successfully! Check details in the card above.")

    except Exception as e:
        st.error(f"Prediction Error: {e}")
        st.exception(e)

Writing app.py


In [5]:
!pkill -f 'streamlit run app.py' || true

^C


In [6]:
!nohup streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false > /tmp/streamlit.log 2>&1 &

In [7]:
import time
time.sleep(3)
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Thu, 01 Oct 2026 01:41:27 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7260
last-modified: Thu, 01 Oct 2026 01:40:14 GMT
etag: "39ba10e3f4b1d66a7b38faef93cb9fa0"
cache-control: no-cache



In [8]:
from google.colab import output
from IPython.display import IFrame, display

url = output.eval_js('google.colab.kernel.proxyPort(8501)')
print(url)
display(IFrame(src=url, width='100%', height=900))

https://8501-m-s-kkb-ass1c2-2mh4a65d7cxy7-c.asia-southeast1-2.prod.colab.dev
